In [ ]:
import pandas
import pandas_gbq
import os

#SQL Query code for your cardiomyopathy cohort goes here

cardiomyo_df.head(5)
print(len(cardiomyo_df))

In [ ]:
import pandas
import pandas_gbq
import os

#SQL Query code for your general population control cohort goes here

generalpop_df.head(5)
print(len(generalpop_df))

In [ ]:
#REQUIREMENTS: 
#  1.Two cohort dataframes must be defined before this and with these variable names: cardiomyo_df AND generalpop_df
#    These cohorts must include only patients who have short read WGS available. Cohort query code can be retrieved by saving a data snapshot
#    for the cohort and copying the Python IPYNB query code Verily provides, then rename the data frame variable names to the names listed above.
#  2. Follow instructions in bag3_to_vcf.ipynb to create the VCF file. This must be done in an AoU Spark Cluster app instance. 
#    And the VCF should be saved to a Google Cloud Bucket, which you can create in the Resources section, called "bucket_for_storage_across_apps".
#  3. Follow instructions in retrieving_vcf_and_tbi.ipynb in a separate, non-spark-cluster app instance notebook 
#    (the same app instance that you're running this file in).
#  4. Ideally, you should use a CPU with at least 16 cores to increase multithreading efficiency.

import pandas as pd
import pysam
import threading

#Add variants here in this ex. VCF format
variant_input = [
    "chr10:119670133:G>A",
    "chr10:119672373:C>T",
    # add variants here
]

#Parsing input variants
def parse_variant(variant_str):
    chrom, pos, alleles = variant_str.split(':')
    ref, alt = alleles.split('>')
    return chrom, int(pos), ref, alt

parsed = [parse_variant(v) for v in variant_input]
n_variants = len(parsed)
print(f"Total variants to query: {n_variants}")

#Dividing variants into chunks for workers
n_workers = min(n_variants, 22)  # no point having more workers than variants
chunk_size = n_variants // n_workers
chunks = []
for i in range(n_workers):
    start = i * chunk_size
    end = start + chunk_size if i < n_workers - 1 else n_variants
    chunks.append(parsed[start:end])

#multithreading logic
batches = [[] for _ in range(n_workers)]

def worker(n, chunk, batchx):
    vcf = pysam.VariantFile('bag3_full_region.vcf.bgz')
    all_samples = list(vcf.header.samples)
    cardio_ids  = set(cardiomyo_df['person_id'].astype(str))
    general_ids = set(generalpop_df['person_id'].astype(str))
    cardio_indices  = [i for i, s in enumerate(all_samples) if s in cardio_ids]
    general_indices = [i for i, s in enumerate(all_samples) if s in general_ids]

    for chrom, pos, ref, alt in chunk:
        cardio_het = cardio_hom = general_het = general_hom = 0

        for rec in vcf.fetch(chrom, pos - 1, pos):
            if rec.pos != pos:
                continue
            try:
                alt_idx = list(rec.alts).index(alt) + 1
            except ValueError:
                continue

            for cohort, indices in [('cardio', cardio_indices),
                                     ('general', general_indices)]:
                for i in indices:
                    gt = rec.samples[all_samples[i]]['GT']
                    a1 = gt[0] if gt[0] is not None else 0
                    a2 = gt[1] if gt[1] is not None else 0
                    carries = (a1 == alt_idx) or (a2 == alt_idx)
                    both    = (a1 == alt_idx) and (a2 == alt_idx)
                    if cohort == 'cardio':
                        if both:       cardio_hom += 1
                        elif carries:  cardio_het += 1
                    else:
                        if both:       general_hom += 1
                        elif carries:  general_het += 1

        batchx.append({
            'position':     pos,
            'ref':          ref,
            'alt':          alt,
            'cardio_het':   cardio_het,
            'cardio_hom':   cardio_hom,
            'general_het':  general_het,
            'general_hom':  general_hom,
        })
        print(f"worker {n}: done {chrom}:{pos} {ref}>{alt}")

    vcf.close()

# Launching workerss
threads = []
for n in range(1, n_workers + 1):
    t = threading.Thread(target=worker, args=(n, chunks[n-1], batches[n-1]))
    t.daemon = True
    t.start()
    threads.append(t)

for t in threads:
    t.join()

print("\nAll workers finished.")

#Save results to .csv
results_df = pd.concat([pd.DataFrame(batch) for batch in batches], ignore_index=True)
results_df = results_df.sort_values('position').reset_index(drop=True)

results_df.to_csv('variant_carrier_counts.csv', index=False)
print(results_df.to_string(index=False))
print("\nSaved to variant_carrier_counts.csv")